### Phase 3 — Extract Skills from Job Descriptions

### Step 1 — build a curated skill dictionary 
this is the most important step in the whole project. Keyword-matching quality determines everything downstream.

In [3]:
import pandas as pd

postings = pd.read_csv("../data/raw/postings.csv")

title_keywords = ["marketing analyst", "data analyst", "marketing data analyst",
                    "digital analyst", "insights analyst", "business analyst"]
pattern = "|".join(title_keywords)
relevant = postings[postings["title"].str.contains(pattern, case=False, na=False)]

target_locations = ["Canada", "Australia"]
location_pattern = "|".join(target_locations)
relevant_geo = relevant[relevant["location"].str.contains(location_pattern, case=False, na=False)]

if len(relevant_geo) < 500:
    target_locations = ["Canada", "Australia", "United States"]
    location_pattern = "|".join(target_locations)
    relevant_geo = relevant[relevant["location"].str.contains(location_pattern, case=False, na=False)]

print(f"Relevant postings: {len(relevant_geo):,}")

Relevant postings: 188


In [4]:
skill_dictionary = {
    "Python": [r"\bpython\b"],
    "SQL": [r"\bsql\b", r"mysql", r"postgresql"],
    "Excel": [r"\bexcel\b"],
    "Tableau": [r"\btableau\b"],
    "Power BI": [r"power\s?bi"],
    "R": [r"\br programming\b", r"\br statistical\b"],
    "Google Analytics": [r"google analytics", r"\bga4\b"],
    "SAS": [r"\bsas\b"],
    "Looker": [r"\blooker\b"],
    "A/B Testing": [r"a/b test", r"ab test", r"split test"],
    "Machine Learning": [r"machine learning", r"\bml\b(?!\w)"],
    "Statistics": [r"\bstatistics\b", r"statistical analysis"],
    "Marketing Mix Modeling": [r"marketing mix model", r"\bmmm\b"],
    "Segmentation": [r"segmentation", r"customer segment"],
    "CRM": [r"\bcrm\b", r"salesforce", r"hubspot"],
    "Data Visualization": [r"data visuali[sz]ation"],
    "Google Ads": [r"google ads", r"adwords"],
    "Facebook Ads": [r"facebook ads", r"meta ads"],
    "SEO": [r"\bseo\b"],
    "ETL": [r"\betl\b"],
    "Big Query": [r"bigquery", r"big query"],
    "Snowflake": [r"snowflake"],
}

import re

def extract_skills(text, skill_dict):
    if pd.isna(text):
        return []
    text_lower = text.lower()
    found = []
    for skill, patterns in skill_dict.items():
        if any(re.search(p, text_lower) for p in patterns):
            found.append(skill)
    return found

relevant_geo = relevant_geo.copy()
relevant_geo["extracted_skills"] = relevant_geo["description"].apply(lambda x: extract_skills(x, skill_dictionary))
relevant_geo["n_skills_found"] = relevant_geo["extracted_skills"].apply(len)

print(f"Postings with at least one skill matched: {(relevant_geo['n_skills_found'] > 0).sum():,} of {len(relevant_geo):,}")

Postings with at least one skill matched: 125 of 188


In [5]:
print(relevant_geo["location"].str.extract(r"(Canada|Australia|United States)", expand=False).value_counts())

location
United States    187
Canada             1
Name: count, dtype: int64


### Step 2 — sanity check: 
look at a few postings with ZERO skills matched — this tells us if our dictionary is missing something important.

In [6]:
zero_skill_postings = relevant_geo[relevant_geo["n_skills_found"] == 0]
print(f"{len(zero_skill_postings)} postings matched no skills — sample descriptions:")
for desc in zero_skill_postings["description"].head(3):
    print(desc[:300])
    print("---")

63 postings matched no skills — sample descriptions:
Job Description: JOB SUMMARY:Responsible for cleaning, analyzing, interpreting, and displaying data using different approaches and business intelligence tools known as data analysis. Responsible for supporting a select group of digital tools and solutions and managing the implementation, business pr
---
As a Partner Automation Specialist at Zoom, you will be responsible for the seamless implementation of technology solutions, both internally within Zoom and externally with our network of distributors, resellers, and referral partners. Your primary focus will be on API integration, managing upgrades
---
Job Title: Entry Level Business Analyst / Product Owner U.S. Citizens and those authorized to work in the U.S. are encouraged to apply. We are able to sponsor at this time.We are a US equal employment opportunity employer. Job Description:Entry Level expertise in gathering, analyzing, and documentin
---


In [7]:
print(f"Current relevant_geo size: {len(relevant_geo):,}")
print(relevant_geo["location"].str.extract(r"(Canada|Australia|United States)", expand=False).value_counts())

Current relevant_geo size: 188
location
United States    187
Canada             1
Name: count, dtype: int64


In [9]:
target_locations = ["Canada", "Australia", "United States"]
location_pattern = "|".join(target_locations)
relevant_geo = relevant[relevant["location"].str.contains(location_pattern, case=False, na=False)]

print(f"Broadened relevant_geo size: {len(relevant_geo):,}")

Broadened relevant_geo size: 188


In [10]:
import requests
import sys
sys.path.append("..")
from config import ADZUNA_APP_ID, ADZUNA_APP_KEY

test_url = "https://api.adzuna.com/v1/api/jobs/ca/search/1"
test_params = {
    "app_id": ADZUNA_APP_ID,
    "app_key": ADZUNA_APP_KEY,
    "what": "data analyst",
    "results_per_page": 5,
    "content-type": "application/json",
}
response = requests.get(test_url, params=test_params, timeout=30)
print(f"Status code: {response.status_code}")
if response.status_code == 200:
    print(f"Results found: {len(response.json().get('results', []))}")
else:
    print(response.text[:500])

Status code: 200
Results found: 5


### Full Canada + Australia Fetch

In [11]:
import requests
import pandas as pd
import sys
sys.path.append("..")
from config import ADZUNA_APP_ID, ADZUNA_APP_KEY

def fetch_adzuna_jobs(country_code, search_terms, max_pages=5, results_per_page=50):
    all_results = []
    for term in search_terms:
        for page in range(1, max_pages + 1):
            url = f"https://api.adzuna.com/v1/api/jobs/{country_code}/search/{page}"
            params = {
                "app_id": ADZUNA_APP_ID,
                "app_key": ADZUNA_APP_KEY,
                "what": term,
                "results_per_page": results_per_page,
                "content-type": "application/json",
            }
            response = requests.get(url, params=params, timeout=30)
            if response.status_code != 200:
                print(f"  {country_code}/{term} page {page}: status {response.status_code}, stopping")
                break

            data = response.json()
            results = data.get("results", [])
            if not results:
                print(f"  {country_code}/{term}: no more results after page {page-1}")
                break

            for job in results:
                all_results.append({
                    "title": job.get("title"),
                    "location": job.get("location", {}).get("display_name"),
                    "description": job.get("description"),
                    "company": job.get("company", {}).get("display_name"),
                    "salary_min": job.get("salary_min"),
                    "salary_max": job.get("salary_max"),
                    "country_code": country_code,
                })

    return pd.DataFrame(all_results)

search_terms = ["marketing analyst", "data analyst", "marketing data analyst", "digital analyst"]

print("Fetching Canada postings...")
canada_jobs = fetch_adzuna_jobs("ca", search_terms, max_pages=5)

print("\nFetching Australia postings...")
australia_jobs = fetch_adzuna_jobs("au", search_terms, max_pages=5)

print(f"\nCanada postings fetched: {len(canada_jobs):,}")
print(f"Australia postings fetched: {len(australia_jobs):,}")

Fetching Canada postings...
  ca/marketing analyst page 2: status 503, stopping
  ca/marketing data analyst: no more results after page 1
  ca/digital analyst: no more results after page 2

Fetching Australia postings...
  au/marketing analyst page 1: status 503, stopping
  au/marketing data analyst: no more results after page 1
  au/digital analyst: no more results after page 3

Canada postings fetched: 384
Australia postings fetched: 369


Check for duplicates (the same job can appear across multiple search terms, e.g. "data analyst" and "marketing data analyst" might both surface the same posting):

In [12]:
canada_before = len(canada_jobs)
canada_jobs = canada_jobs.drop_duplicates(subset=["title", "company", "location"])
print(f"Canada: {canada_before} -> {len(canada_jobs)} after removing duplicates")

australia_before = len(australia_jobs)
australia_jobs = australia_jobs.drop_duplicates(subset=["title", "company", "location"])
print(f"Australia: {australia_before} -> {len(australia_jobs)} after removing duplicates")

Canada: 384 -> 316 after removing duplicates
Australia: 369 -> 349 after removing duplicates


Combine with our Kaggle data:

In [13]:
relevant_geo["source"] = "kaggle_us"
canada_jobs["source"] = "adzuna_ca"
australia_jobs["source"] = "adzuna_au"

combined = pd.concat([relevant_geo, canada_jobs, australia_jobs], ignore_index=True)
print(combined["source"].value_counts())

source
adzuna_au    349
adzuna_ca    316
kaggle_us    188
Name: count, dtype: int64


Check the description-length comparison (important before trusting any cross-market skill comparison):

In [14]:
print(combined.groupby("source")["description"].apply(lambda x: x.astype(str).str.len().mean()).round(0))

source
adzuna_au     500.0
adzuna_ca     498.0
kaggle_us    2773.0
Name: description, dtype: float64
